In [1]:
import pandas as pd

In [2]:
orders = pd.read_csv("olist_orders_dataset.csv")
items = pd.read_csv("olist_order_items_dataset.csv")
customers = pd.read_csv("olist_customers_dataset.csv")
products = pd.read_csv("olist_products_dataset.csv")
reviews = pd.read_csv("olist_order_reviews_dataset.csv")
translation = pd.read_csv("product_category_name_translation.csv")

In [3]:
for name, df in zip(['orders','items','customers','products','reviews','translation'],
                    [orders, items, customers, products, reviews, translation]):
    print(f"\n--- {name} ---")
    print(df.isnull().sum())


--- orders ---
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

--- items ---
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

--- customers ---
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

--- products ---
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm         

In [4]:
orders = orders.dropna(subset=['order_approved_at'])
#Drops every cancelled order

In [5]:
products['product_category_name'] = products['product_category_name'].fillna('unknown')
#fills blank products with unknown
#product_category_name         610
#product_name_lenght           610
#product_description_lenght    610
#product_photos_qty            610
#Since all of these share the same "trait" i also have to replace the nulls
products['product_name_lenght'] = products['product_name_lenght'].fillna(0)
products['product_description_lenght'] = products['product_description_lenght'].fillna(0)
products['product_photos_qty'] = products['product_photos_qty'].fillna(0)

In [6]:
for col in ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']:
    products[col] = products[col].fillna(products[col].median())
#replaces null with median

In [7]:
reviews['review_comment_title'] = reviews['review_comment_title'].fillna('')
reviews['review_comment_message'] = reviews['review_comment_message'].fillna('')
#normal to not have reviews, replace it with "nothing"/empty space

In [9]:
for name, df in zip(['orders','items','customers','products','reviews','translation'],
                    [orders, items, customers, products, reviews, translation]):
    print(f"\n--- {name} ---")
    print(df.dtypes)


--- orders ---
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

--- items ---
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object

--- customers ---
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

--- products ---
product_id                     object
product_category_name          object
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty        

In [10]:
date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

In [11]:
items['shipping_limit_date'] = pd.to_datetime(items['shipping_limit_date'], errors='coerce')

In [12]:
reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'], errors='coerce')
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'], errors='coerce')

In [16]:
orders = orders[orders['order_status'] == 'delivered']
print(f"Rows after filtering: {len(orders)}")

Rows after filtering: 96464


In [18]:
products = products.merge(translation, on='product_category_name', how='left')
#Add the english name column onto products

In [23]:
print(products[['product_category_name', 'product_category_name_english']].head(10))

   product_category_name product_category_name_english
0             perfumaria                     perfumery
1                  artes                           art
2          esporte_lazer                sports_leisure
3                  bebes                          baby
4  utilidades_domesticas                    housewares
5  instrumentos_musicais           musical_instruments
6             cool_stuff                    cool_stuff
7       moveis_decoracao               furniture_decor
8       eletrodomesticos               home_appliances
9             brinquedos                          toys


In [24]:
print(f"\nNulls in english column: {products['product_category_name_english'].isnull().sum()}")


Nulls in english column: 623


In [25]:
print(products[products['product_category_name_english'].isnull()]['product_category_name'].value_counts())
#shows theres 3 products that doesnt have a translation "Unknown" included

product_category_name
unknown                                          610
portateis_cozinha_e_preparadores_de_alimentos     10
pc_gamer                                           3
Name: count, dtype: int64


In [28]:
products['product_category_name_english'] = products['product_category_name_english'].fillna(
    products['product_category_name']
)
#Fills unknowns

In [29]:
products['product_category_name_english'] = products['product_category_name_english'].replace({
    'pc_gamer': 'pc_gamer',
    'portateis_cozinha_e_preparadores_de_alimentos': 'portable_kitchen_appliances'
})
#replaces the 2 products that did not have a name translated

In [30]:
print(products['product_category_name_english'].isnull().sum())

0


In [31]:
# delivery days — how long it took to deliver
orders['delivery_days'] = (
    orders['order_delivered_customer_date'] -
    orders['order_purchase_timestamp']
).dt.days

# is_late — was it delivered after the estimated date?
orders['is_late'] = (
    orders['order_delivered_customer_date'] >
    orders['order_estimated_delivery_date']
).astype(int)

# extract year and month for trend analysis
orders['order_year'] = orders['order_purchase_timestamp'].dt.year
orders['order_month'] = orders['order_purchase_timestamp'].dt.month

/tmp/ipykernel_560/3759764995.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  orders['delivery_days'] = (
/tmp/ipykernel_560/3759764995.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  orders['is_late'] = (
/tmp/ipykernel_560/3759764995.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#retu

In [34]:
print(orders[['delivery_days', 'is_late', 'order_year', 'order_month']].head())
#Verify it works

   delivery_days  is_late  order_year  order_month
0            8.0        0        2017           10
1           13.0        0        2018            7
2            9.0        0        2018            8
3           13.0        0        2017           11
4            2.0        0        2018            2


In [36]:
# revenue per item
items['revenue'] = items['price'] + items['freight_value']
print(items[['price', 'freight_value', 'revenue']].head())
#verify it works

    price  freight_value  revenue
0   58.90          13.29    72.19
1  239.90          19.93   259.83
2  199.00          17.87   216.87
3   12.99          12.79    25.78
4  199.90          18.14   218.04


In [37]:
orders.to_csv("olist_orders_clean.csv", index=False, encoding='utf-8-sig')
items.to_csv("olist_items_clean.csv", index=False, encoding='utf-8-sig')
customers.to_csv("olist_customers_clean.csv", index=False, encoding='utf-8-sig')
products.to_csv("olist_products_clean.csv", index=False, encoding='utf-8-sig')
reviews.to_csv("olist_reviews_clean.csv", index=False, encoding='utf-8-sig')

In [39]:
# FIXING SQL DATA ERROR OF NULLS

In [40]:
print(orders['order_delivered_customer_date'].isnull().sum())

8


In [41]:
orders = orders.dropna(subset=['order_delivered_customer_date'])
#Dropping the 8 nulls of order_delivered

Rows after dropping: 96456


In [42]:
orders.to_csv("olist_orders_cleaned.csv", index=False, encoding='utf-8-sig')